# Kafka to Delta Lake Streaming Upsert (MinIO S3)

This notebook demonstrates how to:
1. Read Protobuf-serialized messages from Kafka (`global_financial_transactions`).
2. Strip Confluent Schema Registry's 5-byte header.
3. Decode the Protocol Buffer using `from_protobuf` and the compiled descriptor file.
4. Upsert (MERGE INTO) incoming transactions into Delta Lake stored in MinIO (`s3a://staging/delta/financial_transactions`).

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr
from pyspark.sql.protobuf.functions import from_protobuf
from delta.tables import DeltaTable

minio_user = os.getenv("MINIO_ROOT_USER", "admin")
minio_password = os.getenv("MINIO_ROOT_PASSWORD", "password")

spark = (SparkSession.builder
    .appName("KafkaToDeltaInteractive")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", minio_user)
    .config("spark.hadoop.fs.s3a.secret.key", minio_password)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark Session initialized with Delta Lake and MinIO!")

## 1. Subscribe to Kafka Stream

In [ ]:
kafka_bootstrap_servers = "kafka-0:29092,kafka-1:29092,kafka-2:29092"
topic_name = "global_financial_transactions"

kafka_stream = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", topic_name)
    .option("startingOffsets", "earliest")
    .load())

# Strip Confluent Schema Registry 5-byte header
raw_payload_df = kafka_stream.select(
    col("key").cast("string").alias("kafka_key"),
    col("timestamp").alias("kafka_timestamp"),
    expr("substring(value, 6, length(value) - 5)").alias("raw_protobuf")
)

# Parse Protobuf binary
desc_path = "/notebooks/transaction.desc"
parsed_df = raw_payload_df.select(
    col("kafka_key"),
    col("kafka_timestamp"),
    from_protobuf(col("raw_protobuf"), "finance.Transaction", descFilePath=desc_path).alias("payload")
)

print("Stream parsing configured successfully!")

## 2. Define Delta Lake Upsert (MERGE) Logic

In [ ]:
def upsert_to_delta(micro_batch_df, batch_id):
    if micro_batch_df.rdd.isEmpty():
        return

    cleaned_df = micro_batch_df.select(
        col("payload.transaction_id").alias("transaction_id"),
        col("payload.timestamp").alias("timestamp"),
        col("payload.amount").alias("amount"),
        col("payload.currency").alias("currency"),
        col("payload.merchant").alias("merchant"),
        col("payload.transaction_type").alias("transaction_type"),
        col("payload.is_flagged_fraud").alias("is_flagged_fraud"),
        col("payload.device_type").alias("device_type"),
        col("payload.user.user_id").alias("user_id"),
        col("payload.user.full_name").alias("user_full_name"),
        col("payload.user.email").alias("user_email"),
        col("payload.user.ip_address").alias("user_ip"),
        col("payload.location.country").alias("country"),
        col("payload.location.city").alias("city"),
        col("payload.location.latitude").alias("latitude"),
        col("payload.location.longitude").alias("longitude"),
        col("kafka_timestamp")
    ).filter("transaction_id IS NOT NULL").dropDuplicates(["transaction_id"])

    delta_path = "s3a://staging/delta/financial_transactions"

    if DeltaTable.isDeltaTable(spark, delta_path):
        delta_table = DeltaTable.forPath(spark, delta_path)
        (delta_table.alias("target")
            .merge(
                cleaned_df.alias("source"),
                "target.transaction_id = source.transaction_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"[Batch {batch_id}] Successfully merged records into Delta table.")
    else:
        cleaned_df.write.format("delta").mode("overwrite").save(delta_path)
        print(f"[Batch {batch_id}] Initialized new Delta table at {delta_path}.")

## 3. Execute Structured Streaming Query (AvailableNow Trigger)

In [ ]:
checkpoint_path = "s3a://staging/delta/checkpoints/financial_transactions"

query = (parsed_df.writeStream
    .format("delta")
    .foreachBatch(upsert_to_delta)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()
print("Finished processing available Kafka events into Delta Lake!")

## 4. Query the Delta Table from MinIO

In [ ]:
delta_df = spark.read.format("delta").load("s3a://staging/delta/financial_transactions")
delta_df.show(10, truncate=False)
print(f"Total records in Delta table: {delta_df.count()}")